# Exploratory Analysis

Narrative EDA only. Every reusable function lives in `src/ecg/` — this
notebook imports them rather than defining anything, so nothing here can
drift out of sync with what the pipeline actually runs.

Run `scripts/01_build_dataset.py` first.


In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from ecg.config import ARTIFACT_DIR, FEATURE_LEAD_INDEX, SOURCE_FS_HZ
from ecg.data.wfdb import EcgDataset
from ecg.features import extract_features
from ecg.viz import SERIES_COLORS, apply_theme


## Load the cached dataset


In [ ]:
cached = np.load(ARTIFACT_DIR / 'dataset.npz', allow_pickle=True)
metadata = pd.DataFrame(json.loads(str(cached['metadata'])))
dataset = EcgDataset(signals=cached['signals'], metadata=metadata)

print(f'{len(dataset)} records, signals {dataset.signals.shape}')
metadata.head()


## How many labels does a record carry?

Multi-label, not multi-class — the distribution below is the reason macro
and micro F1 diverge so sharply later on.


In [ ]:
counts = metadata['n_labels'].fillna(0).astype(int).value_counts().sort_index()

fig = go.Figure(go.Bar(x=counts.index, y=counts.values,
                       marker_color=SERIES_COLORS['default']))
apply_theme(fig, 'Most Records Carry One or Two Diagnoses',
            'Number of SNOMED-CT codes per recording',
            x_title='Labels per record', y_title='Records')
fig.show()


## The long tail of conditions

A handful of rhythms dominate; most conditions appear in a tiny fraction
of records. Any model will look good on micro-averaged metrics and poor on
macro-averaged ones purely because of this shape.


In [ ]:
exploded = metadata['dx_condition_list'].explode().dropna()
top = exploded.value_counts().head(25)

fig = go.Figure(go.Bar(x=top.index, y=top.values,
                       marker_color=SERIES_COLORS['default']))
fig.update_xaxes(tickangle=45)
apply_theme(fig, 'A Few Rhythms Account for Most of the Data',
            'The 25 most frequent conditions',
            y_title='Records')
fig.show()

print(f'{exploded.nunique()} distinct conditions in total')
print(f'top 25 cover {top.sum() / len(exploded):.1%} of all label occurrences')


## What a single record looks like


In [ ]:
record_idx = 0
signal = dataset.lead(record_idx, FEATURE_LEAD_INDEX)
seconds = np.arange(signal.size) / SOURCE_FS_HZ

fig = go.Figure(go.Scatter(x=seconds, y=signal, mode='lines',
                           line=dict(width=1.1, color=SERIES_COLORS['default'])))
apply_theme(fig, 'A Single Lead II Trace',
            f"Record {metadata.iloc[record_idx]['record_id']} - "
            f"{metadata.iloc[record_idx]['dx_conditions']}",
            x_title='Seconds', y_title='Amplitude')
fig.show()


## Features extracted from that record

The same function the pipeline calls — no reimplementation here.


In [ ]:
features = extract_features(signal, SOURCE_FS_HZ)
pd.Series(features).to_frame('value').round(4)


---

Continue with `scripts/02_extract_features.py` to build the full feature
table, then `scripts/03_train.py --model <name>` to train.
